# Full Golden-Company Validation (Slice 3C)

OwnerLens now resolves any SEC-mapped ticker (3A) and normalizes canonical metrics across companies (3B). This notebook validates the **whole** pipeline — Feature 1 fundamentals plus the Feature 2 Economic Value Lens — across three deliberately different businesses:

* **ADBE** — high-margin software, the full-coverage regression baseline
* **V** — a capital-light payment network with unsupported diluted shares
* **COST** — a low-margin, 52/53-week retailer with very different economics

We answer eight questions:

1. Why ADBE, Visa, and Costco?
2. What generalized cleanly?
3. What did Visa break?
4. What did Costco break?
5. Full vs partial analytical coverage
6. Why unsupported is better than fabricated
7. Are the Feature 2 thresholds business-model independent?
8. What assumptions remain before scaling the universe?

> Requires `OWNER_LENS_SEC_USER_AGENT` for the live SEC calls below.

In [ ]:
import os

from owner_lens import (
    SecClient,
    company_coverage,
    company_output,
    economic_value_summary_from_facts,
    format_coverage_report,
    owner_economics_from_facts,
)

USER_AGENT = os.environ.get('OWNER_LENS_SEC_USER_AGENT', 'OwnerLens admin@example.com')
TICKERS = ['ADBE', 'V', 'COST']

with SecClient(USER_AGENT) as client:
    facts = {t: client.retrieve_company_facts(t).raw_facts for t in TICKERS}

list(facts)

## 1. Why ADBE, Visa, and Costco?

The diversity is intentional. Adobe is the validated high-margin baseline; Visa is capital-light with a different debt/equity shape and no tagged diluted-share denominator; Costco is a low-margin, inventory-heavy retailer on a 52/53-week calendar. If the pipeline survives all three honestly, its abstractions are real rather than Adobe-shaped.

## 2 & 5. What generalized cleanly, and full vs partial coverage

The cross-company coverage report classifies every analytical layer per company and gives a reason for every non-full result. ADBE and COST are FULL; Visa is PARTIAL.

In [ ]:
coverages = [company_coverage(facts[t], ticker=t) for t in TICKERS]
print(format_coverage_report(coverages))

## 3. What did Visa break?

Visa tags no weighted-average diluted-share concept, so free-cash-flow per share is unavailable. That single unsupported input flows only into the per-share analyses; revenue, free cash flow, capital efficiency, and capital-allocation facts stay available. Owner economics degrades to `None` per-share fields rather than failing.

In [ ]:
v = owner_economics_from_facts(facts['V'], ticker='V')[0]
print('Visa latest owner economics:')
print('  diluted_shares :', v.diluted_shares)
print('  fcf_per_share  :', v.fcf_per_share)
print('  free_cash_flow :', v.free_cash_flow)
print('  net_margin     :', round(v.net_margin, 4) if v.net_margin else None)
print('  fcf_margin     :', round(v.fcf_margin, 4) if v.fcf_margin else None)

## 4 & 7. What did Costco break, and are the thresholds business-model independent?

Costco broke nothing structurally: the 52/53-week calendar is recognized and every layer runs. The important test is that its ~4% net margin does not make it look economically weak. The Feature 2 thresholds reason about *change*, *per-share growth*, *capital efficiency*, and *trajectory* — not absolute margin levels — so Costco classifies as a healthy compounder just as Adobe does.

In [ ]:
for t in ['ADBE', 'COST']:
    s = economic_value_summary_from_facts(facts[t], ticker=t)
    print(
        f"{t}: overall={s.overall_economic_value_classification.value} "
        f"fcf/share_cagr={s.long_term_fcf_per_share_cagr} roic={s.latest_roic}"
    )

## 6. Why unsupported is better than fabricated

OwnerLens never substitutes point-in-time shares outstanding for weighted-average diluted shares. Visa's per-share layers report `INSUFFICIENT_DATA`, and the company-level output states exactly which input is unsupported — leaving the user with the genuinely available evidence and no invented classification.

In [ ]:
print(company_output(coverages[1], facts['V'], ticker='V'))

## 8. What assumptions remain before scaling the universe?

- **Weighted-average diluted shares** are unsupported for some filers (Visa), blocking per-share analysis until a separately researched share-count source or definition is added.
- **Per-share FCF-CAGR bands** may need industry-aware calibration before scaling far beyond conventional operating companies (documented, not changed here).
- **No persistence or batch ingestion** exists yet, by design.

The pipeline degrades honestly: every green, absent, unsupported, and insufficient result above is economically truthful. That is the confidence needed before persistence and large-universe ingestion.

*This notebook validates architecture and financial semantics; it is not investment advice.*